# Tidal validation of the double-bounce trend (Belém)

Compares the **double-bounce trend of the wetland** estimated in `pauli_hsi_simple.ipynb` (wetland
map from the change heat map, one value per pass) with the **water level of the tide gauge at
São José de Ribamar** (`georg_dev/tide_ribamar_2026-08-15_to_10-04_UTC.csv`, 1-minute data, UTC).
The tide data covers only the Belém stacks.

Steps:

1. Run `pauli_hsi_simple.ipynb` for Belém and each stack you want to validate (`ROI = "Bel_m"`,
   `STACK = "STA3"`, then `"STA1"`): its export cell writes
   `outputs/Bel_m/polsar/<stack>/wetland_trend.csv` with the **exact acquisition time** of every pass.
2. This notebook reads the tide with the code of `georg_dev/tidal_info.ipynb` (same reading, same
   `water_level` look-up, same 0–3 h delay plot as `tide_ribamar_delay.png`), using the
   acquisition times of the stack scenes, so tide and scenes are aligned to the minute.
3. Then it compares the two: aligned time series, correlation for every delay between 0 and
   `MAX_DELAY_H` hours (travel time of the tide from the gauge to the forest), scatter at the best
   delay and a standardized overlay.

**Caution:** a stack has 7 passes, so a correlation rests on 7 points; with so few points only
|r| above about 0.75 is significant at the 5 % level.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

TOPIC_DIR = next(d for d in (Path.cwd(), *Path.cwd().parents) if (d / "flooded_forest").is_dir())
sys.path.insert(0, str(TOPIC_DIR))
from flooded_forest.core import config

# --- parameters ---
ROI_FOLDER = "Bel_m"                      # output folder of the zone (the tide data is for Belém only)
STACKS = ["STA3", "STA1"]                 # stacks to validate (prefix of the stack folder)
TIDE_CSV = TOPIC_DIR / "georg_dev" / "tide_ribamar_2026-08-15_to_10-04_UTC.csv"
METRIC = "wet_T22_db"                     # trend compared with the tide (see the list printed below)
MAX_DELAY_H = 6.0                         # delays scanned: 0 to MAX_DELAY_H hours before the acquisition
DELAY_STEP_MIN = 15

cfg = config.load_config()
paths = cfg.roi_paths(ROI_FOLDER)
trends = []
for prefix in STACKS:
    for stack_dir in paths.stack_dirs():
        if not stack_dir.name.startswith(prefix):
            continue
        csv = paths.polsar / stack_dir.name / "wetland_trend.csv"
        if csv.exists():
            trends.append(pd.read_csv(csv))
            print(f"read {csv.relative_to(TOPIC_DIR)}")
        else:
            print(f"missing {csv.relative_to(TOPIC_DIR)}: run pauli_hsi_simple.ipynb with ROI = 'Bel_m', STACK = '{prefix}'")
if not trends:
    raise FileNotFoundError("no wetland_trend.csv: run pauli_hsi_simple.ipynb for Belém first")
trends = pd.concat(trends, ignore_index=True)
trends["time_utc"] = pd.to_datetime(trends["time_utc"], utc=True)
trends = trends.sort_values("time_utc").reset_index(drop=True)
METRICS = {
    "wet_T22_db": "wetland T22 (double bounce) [dB]",
    "wet_T22_share": "wetland T22 / span",
    "wet_alpha_median": "wetland median alpha [deg]",
    "wet_H_median": "wetland median entropy H",
    "rest_T22_db": "rest of the scene T22 [dB] (control)",
}
print("metrics:", list(METRICS))
trends[["stack", "date", "time_utc", "wet_T22_db", "wet_T22_share", "wet_alpha_median", "wet_H_median",
        "heat_measure", "wetland_cells"]]

## Tide gauge (code of `georg_dev/tidal_info.ipynb`)

Same code as Georg's notebook; only the file path is the parameter `TIDE_CSV` and the figures are
shown, not saved.

In [ ]:
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

path = Path(TIDE_CSV)
lines = path.read_text(encoding="utf-8-sig").splitlines()

header = next(
    index for index, line in enumerate(lines)
    if "stime" in line and "slevel" in line
)

data = pd.read_csv(
    StringIO("\n".join(lines[header:])),
    sep=None,
    engine="python",
)
data.columns = data.columns.str.strip().str.lstrip("#").str.strip()

print("Columns:", data.columns.tolist())
data["stime"] = pd.to_datetime(data["stime"], utc=True, errors="coerce")
data["slevel"] = pd.to_numeric(data["slevel"], errors="coerce")
data = data.dropna(subset=["stime"]).sort_values("stime")

figure, axis = plt.subplots(figsize=(14, 5))
axis.plot(data["stime"], data["slevel"], linewidth=0.7)
axis.set(
    title="São José de Ribamar — Water level",
    xlabel="Date (UTC)",
    ylabel="Water level (m, station reference)",
)
axis.grid(alpha=0.3)
figure.autofmt_xdate()
figure.tight_layout()
plt.show()

### Water level at the acquisition of the stack scenes

Georg's code took the scene times from the names of old Pauli figures; here they are the exact
acquisition times of the stack products (`time_utc` of `wetland_trend.csv`), so the level is read
at the minute of each scene. `water_level` is unchanged: nearest gauge sample, NaN if more than
5 minutes away.

In [ ]:
scenes = pd.DataFrame(
    {
        "scene": trends["stack"].str.slice(0, 4) + "_" + trends["date"],
        "time_utc": trends["time_utc"],
    }
)


def water_level(time):
    position = data["stime"].searchsorted(time)
    candidates = [
        index for index in (position - 1, position) if 0 <= index < len(data)
    ]
    nearest = min(candidates, key=lambda index: abs(data["stime"].iloc[index] - time))
    if abs(data["stime"].iloc[nearest] - time) > pd.Timedelta("5min"):
        return float("nan")
    return float(data["slevel"].iloc[nearest])


scenes["level_m"] = scenes["time_utc"].map(water_level).round(2)

figure, axis = plt.subplots(figsize=(14, 5))
axis.plot(data["stime"], data["slevel"], linewidth=0.7, zorder=1)
for time in scenes["time_utc"]:
    axis.axvline(time, color="red", alpha=0.35, linewidth=0.8, zorder=2)
axis.scatter(
    scenes["time_utc"],
    scenes["level_m"],
    color="red",
    s=18,
    zorder=3,
    label="Scene acquisition",
)
axis.set(
    title="São José de Ribamar — Water level at scene acquisition",
    xlabel="Date (UTC)",
    ylabel="Water level (m, station reference)",
)
axis.grid(alpha=0.3)
axis.legend(loc="upper right")
figure.autofmt_xdate()
figure.tight_layout()
plt.show()

scenes

### Reference: gauge level 0–3 h before each scene (`tide_ribamar_delay.png`)

In [ ]:
delays_h = [0, 1, 2, 3]
for hours in delays_h:
    scenes[f"level_{hours}h"] = (
        (scenes["time_utc"] - pd.Timedelta(hours=hours)).map(water_level).round(2)
    )

figure, axis = plt.subplots(figsize=(14, 5))
colors = ["0.45", "C0", "C1", "C3"]
for time in scenes["time_utc"]:
    axis.axvline(time, color="0.75", linewidth=0.6, zorder=0)
for hours, color in zip(delays_h, colors):
    axis.scatter(
        scenes["time_utc"],
        scenes[f"level_{hours}h"],
        s=28,
        color=color,
        zorder=3,
        label=f"{hours} h before acquisition",
    )
for _, scene in scenes.iterrows():
    axis.plot(
        [scene["time_utc"]] * len(delays_h),
        [scene[f"level_{hours}h"] for hours in delays_h],
        color="0.7",
        linewidth=0.8,
        zorder=2,
    )
axis.set(
    title="Gauge level 0–3 h before each scene (travel time toward the forest)",
    xlabel="Acquisition (UTC)",
    ylabel="Water level (m, station reference)",
)
axis.grid(alpha=0.3, axis="y")
axis.legend(loc="upper right")
figure.autofmt_xdate()
figure.tight_layout()
plt.show()

scenes[["scene", "time_utc", "level_0h", "level_1h", "level_2h", "level_3h"]]

## Double-bounce trend vs tide

### Aligned time series, one figure per stack

Top: gauge level over the stack period, with the scenes (red) at their acquisition minute.
Bottom: the double-bounce trend (`METRIC`) of the wetland and, as a control, of the rest of the
scene, at the same times.

In [ ]:
db = pd.concat([trends.reset_index(drop=True), scenes.drop(columns=["time_utc"]).reset_index(drop=True)], axis=1)
STACK_COLORS = dict(zip(sorted(db["stack"].unique()), ["#e0302a", "#2a64d6", "#2fa84f"]))

for stack, g in db.groupby("stack"):
    t0, t1 = g["time_utc"].min() - pd.Timedelta("1D"), g["time_utc"].max() + pd.Timedelta("1D")
    tide = data[(data["stime"] >= t0) & (data["stime"] <= t1)][["stime", "slevel"]].copy()
    gaps = tide["stime"].diff() > pd.Timedelta("10min")                 # break the line at gauge gaps
    tide = pd.concat([tide, tide.loc[gaps].assign(stime=lambda x: x["stime"] - pd.Timedelta("1min"), slevel=np.nan)]
                     ).sort_values("stime")
    fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(14, 6.5), sharex=True, height_ratios=[1.2, 1], layout="constrained")
    ax0.plot(tide["stime"], tide["slevel"], color="#7a7a7a", lw=0.8)
    ax0.scatter(g["time_utc"], g["level_0h"], color="#e0302a", s=30, zorder=3, label="level at acquisition")
    for t in g.loc[g["level_0h"].isna(), "time_utc"]:
        ax0.annotate("no gauge data", (t, ax0.get_ylim()[0]), xytext=(3, 5), textcoords="offset points",
                     fontsize=7, color="#e0302a", rotation=90)
    ax0.set_ylabel("water level [m]")
    ax0.legend(frameon=False, loc="upper right")
    ax0.set_title(f"{stack}: São José de Ribamar water level and scene acquisitions", fontsize=10)
    ax1.plot(g["time_utc"], g[METRIC], "-o", color="#e0302a", lw=2, ms=6, label=METRICS[METRIC])
    if METRIC.startswith("wet_") and METRIC.replace("wet_", "rest_") in g:
        ax1.plot(g["time_utc"], g[METRIC.replace("wet_", "rest_")], "--s", color="#7a7a7a", lw=1.5, ms=5,
                 label=METRICS.get(METRIC.replace("wet_", "rest_"), "rest of the scene (control)"))
    ax1.set_ylabel(METRIC)
    ax1.legend(frameon=False, loc="upper right")
    for ax in (ax0, ax1):
        for t in g["time_utc"]:
            ax.axvline(t, color="#e0302a", alpha=0.25, lw=0.8)
        ax.grid(color="#e4e3dd", lw=0.5)
        ax.spines[["top", "right"]].set_visible(False)
    fig.autofmt_xdate()
    plt.show()

### Correlation vs delay

For every delay d between 0 and `MAX_DELAY_H` hours, the gauge level at (acquisition − d) is
correlated (Pearson r) with each trend metric, per stack. The delay with the strongest correlation
of the wetland double bounce is an estimate of the travel time of the tide from the gauge to the
forest. Grey band: |r| below the 5 % significance level for the stack with most points; passes
without gauge data (gaps in the record) are left out, so a stack can have fewer than 7 points.

**Read the control (last panel):** if the rest of the scene correlates with the tide as strongly as
the wetland, the tide signal is not specific to the wetland map: either the whole scene responds to
the tide (tidal floodplain forest everywhere) or a scene-wide effect follows the same sequence of
dates. A wetland-specific signal shows up as a wetland correlation clearly stronger than the
control. Correlation ignores the size of the swing, so also compare the
amplitudes in the aligned plot above: a control that correlates but moves by a few tenths of a dB
while the wetland moves by 1–2 dB means the tide acts mostly on the wetland.

In [ ]:
delays = np.arange(0, MAX_DELAY_H + 1e-9, DELAY_STEP_MIN / 60)
scan = []
for stack, g in db.groupby("stack"):
    for d in delays:
        level = (g["time_utc"] - pd.Timedelta(hours=float(d))).map(water_level).to_numpy()
        for m in METRICS:
            ok = np.isfinite(level) & np.isfinite(g[m].to_numpy())
            r, p = stats.pearsonr(level[ok], g[m].to_numpy()[ok]) if ok.sum() >= 3 else (np.nan, np.nan)
            scan.append({"stack": stack, "delay_h": d, "metric": m, "r": r, "p": p, "n": int(ok.sum())})
scan = pd.DataFrame(scan)

def r_critical(n):
    """|r| needed for p < 0.05 (two-sided) with n points."""
    t = stats.t.ppf(0.975, n - 2)
    return t / np.sqrt(n - 2 + t ** 2)


n_by_stack = scan.groupby("stack")["n"].max()
n_pts = int(n_by_stack.max())
r_crit = r_critical(n_pts)
fig, axes = plt.subplots(1, len(METRICS), figsize=(4 * len(METRICS), 3.6), sharey=True, layout="constrained")
for ax, m in zip(axes, METRICS):
    ax.axhspan(-r_crit, r_crit, color="#e4e3dd", lw=0)
    for stack, g in scan[scan["metric"] == m].groupby("stack"):
        ax.plot(g["delay_h"], g["r"], color=STACK_COLORS[stack], lw=2, label=stack[:4])
    ax.axhline(0, color="#7a7a7a", lw=0.8)
    ax.set_title(METRICS[m], fontsize=8)
    ax.set_xlabel("delay before acquisition [h]")
    ax.grid(color="#e4e3dd", lw=0.5)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Pearson r with the gauge level")
axes[0].set_ylim(-1, 1)
axes[-1].legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.suptitle("correlation of the trends with the tide vs delay. Grey: |r| < " f"{r_crit:.2f} = not significant for n = {n_pts}; "
             + ", ".join(f"{s[:4]}: n = {n} (|r| > {r_critical(n):.2f} needed)" for s, n in n_by_stack.items()), fontsize=10)
plt.show()

best = (scan[scan["metric"] == METRIC].assign(abs_r=lambda s: s["r"].abs())
        .sort_values("abs_r", ascending=False).groupby("stack").head(1)[["stack", "delay_h", "r", "p", "n"]])
print(f"best delay for {METRIC}:")
best

### Correlation table at the reference delays 0–3 h

Pearson r (linear) and Spearman ρ (rank, robust to one odd point) with their p-values.

In [ ]:
table = []
for stack, g in db.groupby("stack"):
    for hours in delays_h:
        level = g[f"level_{hours}h"].to_numpy()
        for m in METRICS:
            ok = np.isfinite(level) & np.isfinite(g[m].to_numpy())
            r, p = stats.pearsonr(level[ok], g[m].to_numpy()[ok])
            rho, p_rho = stats.spearmanr(level[ok], g[m].to_numpy()[ok])
            table.append({"stack": stack[:4], "delay_h": hours, "metric": m, "pearson_r": r, "p": p,
                          "spearman_rho": rho, "p_rho": p_rho, "n": int(ok.sum())})
table = pd.DataFrame(table)
table.pivot_table(index=["metric", "delay_h"], columns="stack", values=["pearson_r", "p"]).round(3)

### Scatter and standardized overlay at the best delay

Left: `METRIC` against the gauge level at the best delay of each stack, with the least-squares
line. Right: both series standardized (z-score: mean 0, standard deviation 1) on a common axis, so
their shapes can be compared pass by pass.

In [ ]:
for _, b in best.iterrows():
    g = db[db["stack"] == b["stack"]]
    level = (g["time_utc"] - pd.Timedelta(hours=float(b["delay_h"]))).map(water_level).to_numpy()
    y = g[METRIC].to_numpy()
    ok = np.isfinite(level) & np.isfinite(y)
    slope, intercept = np.polyfit(level[ok], y[ok], 1)
    color = STACK_COLORS[b["stack"]]

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4), width_ratios=[1, 1.6], layout="constrained")
    ax0.scatter(level, y, color=color, s=50, zorder=3)
    for lv, yy, date in zip(level, y, g["date"]):
        ax0.annotate(date[5:], (lv, yy), textcoords="offset points", xytext=(5, 4), fontsize=7, color="#555555")
    xs = np.linspace(np.nanmin(level), np.nanmax(level), 50)
    ax0.plot(xs, slope * xs + intercept, color="#7a7a7a", lw=1.5,
             label=f"r = {b['r']:.2f}, p = {b['p']:.3f}\nslope {slope:.3g} per m")
    ax0.set_xlabel(f"gauge level {b['delay_h']:g} h before acquisition [m]")
    ax0.set_ylabel(METRICS[METRIC])
    ax0.legend(frameon=False)

    z = lambda v: (v - np.nanmean(v)) / np.nanstd(v)
    ax1.plot(g["time_utc"], z(level), "-o", color="#7a7a7a", lw=2, ms=6, label=f"gauge level −{b['delay_h']:g} h (z)")
    ax1.plot(g["time_utc"], z(y), "-o", color=color, lw=2, ms=6, label=f"{METRIC} (z)")
    ax1.axhline(0, color="#b0b0a8", lw=0.8)
    ax1.set_ylabel("standardized value (z-score)")
    ax1.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    for ax in (ax0, ax1):
        ax.grid(color="#e4e3dd", lw=0.5)
        ax.spines[["top", "right"]].set_visible(False)
    fig.autofmt_xdate()
    fig.suptitle(f"{b['stack']}: {METRIC} vs tide at the best delay ({b['delay_h']:g} h)", fontsize=10)
    plt.show()